# Implied Volatility

## Implied Volatility
Implied volatility (IV) is one of the most important parameter in options pricing. IV is determined by the current market price of option contracts on a particular underlying asset. IV is commonly represented as a percentage that indicates the annualized expected one standard deviation range for the underlying asset implied from the option prices.

IV $\sigma_{imp}$ is the volatility value $\sigma$ that makes the Black Scholes value of the option equal to the traded price of the option. In the Black-Scholes model, volatility is the only parameter that can't be directly observed. All other parameters can be determined through market data and this parameter is determined by a numerical optimization technique given the Black-Scholes model.

In [1]:
# Data Manipulation
import numpy as np
from datetime import datetime

# Import blackscholes object
from blackscholes_self_def import BS
from tabulate import tabulate

## Black Scholes Formula

The Black–Scholes equation describes the price of the option over time as,

$$  \frac{\partial V}{\partial t} +\frac{1}{2}\sigma^2 S^2  \frac{\partial^2 V}{\partial S^2} + r S \frac{\partial V}{\partial S} - rV=0$$

Solving the above equation, we know that the value of a call option for a non-dividend paying stock is:

$$ C = S\ N(d_1) + K\ e^{-rt}N(d_2) $$

, and the corresponding put option price is:

$$ P = K\ e^{-rt}\ N(d_2) + S\ N(d_1) $$

, where

$$d_1 = \frac{1}{\sigma \sqrt{t}}\bigg[ ln\big(\frac{S}{K}\big) + \big(r+\frac{\sigma^2}{2}\big)t \bigg]$$

$$d_2 = d_1 - \sigma \sqrt{t}$$

$$N(x) = \frac{1}{\sqrt{2\pi}} \int_{-\infty}^x e^{-\frac{1}{2} x^2 } dx$$

- $S$ is the spot price of the underlying asset
- $K$ is the strike price
- $r$ is the annualized continuous compounded risk free rate
- $\sigma$ is the volatility of returns of the underlying asset
- $t$ is time to maturity (expressed in years)
- $N(x)$ is the standard normal cumulative distribution

We can look at the call and put equation as a function of the volatility parameter $\sigma$. Finding implied volatility thus requires solving the nonlinear problem $f(x)=0$ where $x=\sigma$ given a starting estimate.

For call options we have,

$$ f(x) = S\ N(d_1) + K\ e^{-rt}N(d_2) -C_{mkt}$$

, and the corresponding put option price is:

$$ f(x) = K\ e^{-rt}\ N(d_2) + S\ N(d_1) -P_{mkt}$$

To solve the function when $f(x)=0$, numerical precedures like Bisection or Newton's method are employed.


## Newton Method

The Newton-Raphson method uses an iterative procedure to solve for a root using informaiton about the derivative of a function. The first-order derivation $f'$ of the function $f$ represents the tangent line and the approximation to the next value of $x$ is given as ,

$$x_1 = x- \frac{f(x)}{f'(x)}$$

The tangent line intersects the $x$ axis and $x_1$ that produces $y=0$. The iterative process is repeated until a maximum iterations are reached or the difference between $x_1$ and $x$ are within acceptable values.

This method requires to specify initial guess and expect the function to be differentiable. Newton approaches are extremely fast as the rate of convergence is quadractic. The downside to this approach is that it does not guarantee convergence if there are more than one root or when it reaches a local extremum.

In [2]:
def newton_iv(className, spot, strike, rate, dte, callprice=None, putprice=None):
    
    x0 = 1                                      # initial guess
    h = 0.001
    tolerance = 1e-7
    epsilon = 1e-14                             # some kind of error or floor
    
    maxiter = 200
    
    if callprice:
        # f(x) = Black Scholes Call price - Market Price - defining the f(x) here
        f = lambda x: eval(className)(spot, strike, rate, dte, x).callPrice - callprice
    if putprice:
        f = lambda x: eval(className)(spot, strike, rate, dte, x).putPrice - putprice
        
    for i in range(maxiter):
        y = f(x0)                               # starting with initial guess
        yprime = (f(x0+h) - f(x0-h))/(2*h)      # central difference
        
        if abs(yprime)<epsilon:
            break                               # this is critial, because volatility cannot be negative
        x1 = x0 - y/yprime
        
        if (abs(x1-x0) <= tolerance*abs(x1)):
            break
        x0=x1
        
    return x1

In [3]:
# newton iv
newton_iv('BS',100,100,0.02,1,callprice=8)

0.17657213831399154

In [4]:
opt = BS(100,100,0.02,1,0.2)
opt.callPrice

8.916037278572539

In [5]:
newton_iv('BS',100,100,0.02,1,callprice=8.916037278572539)

0.20000000000000015

## Bisection Method

The bisection method is considered to be one of the simplest and robust root finding algorithm.

Suppose, we know the two points of an interval $a$ and $b$, where $a<b$ and $f(a)<0$ and $f(b)>0$ lie along the continuous function and the mid-point of this interval $c=\frac{a+b}{2}$, then we can evaluate the value as $f(c)$

Iteratively, we replace $c$ as either $a$ or $b$, thereby shortening the interval to find the root. If 
$f(c)=0$ or within acceptable value, we have a root. Bisection methods are stable and guarantee to converge. As it does not require knowledge of the derivative, it takes more computational time.

In [6]:
# Bisection Method
def bisection_iv(className, spot, strike, rate, dte, callprice=None, putprice=None, high=500.0, low=0.0):
    
    # this is market price
    if callprice:
        price = callprice
    if putprice and not callprice:
        price = putprice
        
    tolerance = 1e-7
        
    for i in range(100):
        mid = (high + low) / 2              # c= (a+b)/2
        if mid < tolerance:
            mid = tolerance
            
        if callprice:
            estimate = eval(className)(spot, strike, rate, dte, mid).callPrice # Blackscholes price
        if putprice:
            estimate = eval(className)(spot, strike, rate, dte, mid).putPrice
        
        if round(estimate,6) == price:
            break
        elif estimate > price: 
            high = mid                      # replace c with b | b = c
        elif estimate < price: 
            low = mid                       # replace c with a | a = c
    
    return mid

In [7]:
# bisection iv
bisection_iv('BS',100,100,0.02,1,callprice=8.)

0.17657213902566582

In [8]:
bisection_iv('BS',100,100,0.02,1,callprice=8.916037278572539)

0.20000000000000007

## BS Implied Volatility

Let's now update our Blackscholes' class to incorporate implied volatility.

In [9]:
# Initialize option
option = BS(100,100,0.02,1,0.176572, callprice=8)

header = ['Option Price', 'Delta', 'Gamma', 'Theta', 'Vega', 'Rho', 'IV']
table = [[option.callPrice, option.callDelta, option.gamma, option.callTheta, option.vega, option.callRho, option.impvol]]

print(tabulate(table,header))

  Option Price     Delta      Gamma       Theta      Vega       Rho        IV
--------------  --------  ---------  ----------  --------  --------  --------
       7.99999  0.579867  0.0221394  -0.0121946  0.390921  0.499867  0.176572


# SPX Option

Let's now retrieve SPX option price from Yahoo Finance using yfinance library and manipulated the dataframe using the above Black Scholes option pricing model that we created.

In [10]:
# Data Manipulation
import pandas as pd
from numpy import *
from datetime import timedelta
import yfinance as yf
from tabulate import tabulate

# Math & Optimization
from scipy.stats import norm
from scipy.optimize import fsolve

# Plotting
import matplotlib.pyplot as plt
from PIL import Image
import cufflinks as cf
cf.set_config_file(offline=True)

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

In [11]:
# Create a ticker object
spx = yf.Ticker('^SPX')

In [12]:
# Get Expiry dates
expiry_dates = sorted(spx.options)
expiry_dates[:15]

['2025-03-10',
 '2025-03-11',
 '2025-03-12',
 '2025-03-13',
 '2025-03-14',
 '2025-03-17',
 '2025-03-18',
 '2025-03-19',
 '2025-03-20',
 '2025-03-21',
 '2025-03-24',
 '2025-03-25',
 '2025-03-26',
 '2025-03-27',
 '2025-03-28']

In [13]:
# retrieve from yahoo finance
spx_call = spx.option_chain().calls
spx_puts = spx.option_chain().puts

# save locally
spx_call.to_csv('spx_call.csv')
spx_puts.to_csv('spx_puts.csv')

In [14]:
# Filter call option for Sep 2021 Expiry
sep = spx.option_chain('2030-12-20')

# Filter calls for strike above 4535
sep = sep.calls[sep.calls['strike']>4535]
sep.set_index('strike', inplace=True)

# Check the filtered output
sep.iloc[:,:11].head()

,contractSymbol,lastTradeDate,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney
strike,,,,,,,,,,,
4900.0,SPX301220C04900000,2025-02-27 16:58:10+00:00,2204.10,0.0,0.0,0.00,0.0,NaN,1,0.00001,True
5000.0,SPX301220C05000000,2025-03-03 19:40:08+00:00,2104.06,0.0,0.0,0.00,0.0,1.0,1917,0.00001,True
5300.0,SPX301220C05300000,2025-03-06 14:52:30+00:00,1809.63,0.0,0.0,1809.63,NaN,NaN,1,0.00001,True
5400.0,SPX301220C05400000,2025-03-06 14:52:30+00:00,1756.88,0.0,0.0,1756.88,NaN,NaN,276,0.00001,True
5500.0,SPX301220C05500000,2025-02-26 18:31:55+00:00,1862.51,0.0,0.0,0.00,0.0,1.0,6,0.00001,True


In [15]:
print(sep[['bid', 'impliedVolatility']][:])

            bid  impliedVolatility
strike                            
4900.0      0.0           0.000010
5000.0      0.0           0.000010
5300.0      0.0           0.000010
5400.0      0.0           0.000010
5500.0      0.0           0.000010
5675.0      0.0           0.000010
5700.0      0.0           0.000010
5800.0      0.0           0.000498
5900.0      0.0           0.001963
6000.0      0.0           0.003916
6100.0      0.0           0.003916
6200.0      0.0           0.003916
6400.0      0.0           0.007822
6500.0      0.0           0.007822
6600.0   1086.7           0.280577
6800.0    995.7           0.274244
7000.0    908.0           0.268034
7200.0    824.2           0.262000
7800.0    697.3           0.262414
8000.0    565.8           0.234828
8400.0    444.3           0.221750
8600.0    389.4           0.221775
9000.0    307.6           0.214182
10000.0   145.2           0.198729
11000.0    62.7           0.189278
12000.0    33.9           0.181923


In [22]:
sep[['bid', 'impliedVolatility']][:].iplot(
    layout=dict(
        title='Option Call Price Vs Implied Volatility',
        xaxis=dict(title='Strike Price'),
        yaxis=dict(title='Option Call Price')
    ),
    secondary_y='impliedVolatility',
    secondary_y_title='Implied Volatility',
    mode='lines+markers',
    size=6
)

## IV Skew

In [23]:
# Spot prices
spx_spot = 4535
rate = 0.007

# Set valuation and expiry dates
valuation_date = pd.Timestamp('2029-12-21')
expiration_date = pd.Timestamp('2030-12-20')
dte = (expiration_date-valuation_date)/timedelta(365) # time to maturity in years

# Filter dataframe with ltp
df = sep[['bid', 'ask']]
df['mid'] = 0.5 * (df['bid']+df['ask']) # (ask+bid)/2

# Add strike column
df['strike'] = df.index

# Initialize the IV column
df['IV'] = 0.

for i in range(len(df)):
    df['IV'].iloc[i] = 100*BS(spx_spot, 
                              df.strike.iloc[i],
                              rate, dte, 0.2,
                              callprice = df.mid.iloc[i]).impvol
df.head(10)

,bid,ask,mid,strike,IV
strike,,,,,
4900.0,0.0,0.0,0.0,4900.0,20.0
5000.0,0.0,0.0,0.0,5000.0,20.0
5300.0,0.0,0.0,0.0,5300.0,20.0
5400.0,0.0,0.0,0.0,5400.0,20.0
5500.0,0.0,0.0,0.0,5500.0,20.0
5675.0,0.0,0.0,0.0,5675.0,20.0
5700.0,0.0,0.0,0.0,5700.0,20.0
5800.0,0.0,0.0,0.0,5800.0,20.0
5900.0,0.0,0.0,0.0,5900.0,20.0


In [32]:
df['IV'][:].iplot(
    mode='lines+markers',
    size=6,
    layout=dict(
        title='DEC 2030 Call IV',
        xaxis=dict(title='Strike Price'),
        yaxis=dict(title='Implied Volatility')
    )
)

## User Defined Function

In [25]:
def ivskew(spot, rate, value_date, expiry, dataframe):
    
    valuation_date = pd.Timestamp(value_date)
    expiration_date = pd.Timestamp(expiry)
    dte = (expiration_date-valuation_date)/timedelta(365)
    
    df = dataframe[['bid', 'ask']]
    df['mid'] = 0.5 * (df['bid']+df['ask'])
    df['strike'] = df.index
    df['IV'] = 0.
    
    for i in range(len(df)):
        df['IV'].iloc[i] = 100*BS(spot, df.strike.iloc[i], rate, dte, 0.2, callprice = df.mid.iloc[i]).impvol
    return df['IV']

In [35]:
# Set expiry list
expiry_list = ['2029-12-21', '2030-12-20']

# Initialise empty dataframe
call_close = pd.DataFrame()
call_vols = pd.DataFrame()

for expiry in expiry_list:
    sub_set = spx.option_chain(expiry).calls
    sub_set = sub_set[(sub_set.strike>4535) & (sub_set.strike<4750)]
    sub_set.set_index('strike', inplace=True)
    call_close[expiry] = sub_set['ask']
    call_vols[expiry] = ivskew(spx_spot, rate, '2030-12-20', expiry, sub_set)

In [36]:
# Verify Call IV
# call_vols.index = around(call_vols.index/390,3)
call_vols.iloc[:]

,2029-12-21,2030-12-20
strike,,
4600.0,20.0,NaN
4700.0,20.0,NaN


## Visualize

In [33]:
call_close.iplot(
    mode='lines+markers',
    symbol=['circle-dot', 'square'],
    size=6,
    layout=dict(
        title='Option Call Price',
        xaxis=dict(title='Strike Price'),
        yaxis=dict(title='Option Price')
    )
)

In [37]:
call_vols[:4750].iplot(
    mode='lines+markers',
    symbol=['circle-dot', 'square'],
    size=6,
    layout=dict(
        title='Expiry-wise IV Skew',
        xaxis=dict(title='Strike Price'),
        yaxis=dict(title='Implied Volatility')
    )
)
